In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.svm import SVC
from sklearn.datasets import load_iris


import math

In [44]:
iris = load_iris()

df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = iris.target

target = 'species'
remove_cols = ['']
features = [col for col in df.columns if col != target and col not in remove_cols]

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [45]:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state=42))
])

pipe_svm_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf', random_state=42))
])

pipe_svm_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='linear', random_state=42))
])

pipe_svm_poly = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='poly', random_state=42))
])


kfold = KFold(n_splits=10, shuffle=True, random_state=42)

models = {
    "Logistic Regression": pipe_lr,
    "Random Forest": pipe_rf,
    "Support Vector Machine (SVM) Kernal: rbf": pipe_svm_rbf,
    "Support Vector Machine (SVM) Kernal: linear": pipe_svm_linear,
    "Support Vector Machine (SVM) Kernal: poly": pipe_svm_poly
}
scores = {}
for name, model in models.items():
    print(f"{name}...", end=" ")
    model.fit(X_train, y_train)
    scores[name] = model.score(X_test, y_test)
    print(f"Accuracy: {scores[name]:.4f}")

Logistic Regression... Accuracy: 1.0000
Random Forest... Accuracy: 1.0000
Support Vector Machine (SVM) Kernal: rbf... Accuracy: 1.0000
Support Vector Machine (SVM) Kernal: linear... Accuracy: 0.9667
Support Vector Machine (SVM) Kernal: poly... Accuracy: 0.9667


In [46]:
scores = {}
for name, model in models.items():
    print(f"{name}...", end=" ")
    model.fit(X_train, y_train)
    accuracy_list = cross_val_score(model, X, y, cv=kfold, scoring='accuracy')
    mean_accuracy = np.mean(accuracy_list)
    std_dev = np.std(accuracy_list)
    scores[name] = mean_accuracy
    print(f"Cross Validation Score: {mean_accuracy:.4f} (Std Dev = {std_dev:.4f})")


Logistic Regression... Cross Validation Score: 0.9533 (Std Dev = 0.0670)
Random Forest... Cross Validation Score: 0.9600 (Std Dev = 0.0533)
Support Vector Machine (SVM) Kernal: rbf... Cross Validation Score: 0.9600 (Std Dev = 0.0533)
Support Vector Machine (SVM) Kernal: linear... Cross Validation Score: 0.9600 (Std Dev = 0.0533)
Support Vector Machine (SVM) Kernal: poly... Cross Validation Score: 0.9200 (Std Dev = 0.0718)


In [47]:
best_model_name = max(scores, key=scores.get)
best_model = models[best_model_name]
print(f"\nBest performing model: {best_model_name}")


Best performing model: Random Forest


In [48]:
poly_score_percent = scores['Support Vector Machine (SVM) Kernal: poly'] * 100
is_98_percent = np.isclose(poly_score_percent, 98.0, atol=1.0) # Check if it's within 1%
if is_98_percent:
    print(f"   YES. Using SVC(kernel='poly') resulted in {poly_score_percent:.2f}% accuracy, which is at or very close to 98%.")
else:
    print(f"   NO. Using SVC(kernel='poly') resulted in {poly_score_percent:.2f}% accuracy, which is not 98%.")


   NO. Using SVC(kernel='poly') resulted in 92.00% accuracy, which is not 98%.


In [49]:
import pickle

with open('best_model - iris.pkl', 'wb') as f:
    pickle.dump(best_model, f)

In [50]:
data_to_predict = pd.DataFrame([
    {'sepal length (cm)': 4.7, 'sepal width (cm)': 3.2, 'petal length (cm)': 1.3, 'petal width (cm)': 0.2},
    {'sepal length (cm)': 3, 'sepal width (cm)': 4.6, 'petal length (cm)': 4.6, 'petal width (cm)': 1.5	},
    {'sepal length (cm)': 15, 'sepal width (cm)': 4.6, 'petal length (cm)': 3.1, 'petal width (cm)': 2.5	},
    {'sepal length (cm)': 1, 'sepal width (cm)': 2.5, 'petal length (cm)': 10, 'petal width (cm)': 4.5	},
])

predictions = best_model.predict(data_to_predict)

for sl, sw, pl, pw, prediction in zip(data_to_predict['sepal length (cm)'], data_to_predict['sepal width (cm)'], data_to_predict['petal length (cm)'], data_to_predict['petal width (cm)'], predictions):
    print(f"The Iris with {sl} sepal length (cm), {sw} sepal width (cm), {pl} petal length (cm), {pw} petal width is..:")
    print(f" - {prediction}")


The Iris with 4.7 sepal length (cm), 3.2 sepal width (cm), 1.3 petal length (cm), 0.2 petal width is..:
 - 0
The Iris with 3.0 sepal length (cm), 4.6 sepal width (cm), 4.6 petal length (cm), 1.5 petal width is..:
 - 1
The Iris with 15.0 sepal length (cm), 4.6 sepal width (cm), 3.1 petal length (cm), 2.5 petal width is..:
 - 2
The Iris with 1.0 sepal length (cm), 2.5 sepal width (cm), 10.0 petal length (cm), 4.5 petal width is..:
 - 2
